# Measurement Session Log

Record an electrical measurement session. Links measurements to devices and HDF5 files.

In [1]:
import sys, os
from datetime import datetime
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, PROJECT_ROOT)

from src.database import (
    init_db, list_samples, get_sample, add_step,
    add_measurement, update_step
)
from src.readers import read_hdf5, list_measurements
import pandas as pd

init_db()

samples_all = list_samples()
df = pd.DataFrame(samples_all)
df[['label', 'step_status']]

,label,step_status
0,260702_W1_P1,"photolithography:completed, developing:complet..."
1,260702_W1_P2,"photolithography:completed, developing:complet..."
2,260702_W1_P3,"photolithography:completed, developing:complet..."
3,260702_W1_P4,"photolithography:completed, developing:complet..."
4,260626_W1_P1,"photolithography:completed, evaporation:comple..."
5,260626_W1_P2,"photolithography:completed, evaporation:comple..."
6,260626_W1_P3,"photolithography:completed, evaporation:comple..."
7,260626_W1_P4,"photolithography:completed, evaporation:comple..."


## 1. Select Sample & Device

Which sample are you measuring today?

In [2]:
SAMPLE_LABEL = '260626_W1_P1'  # Change this

matches = [s for s in samples_all if s['label'] == SAMPLE_LABEL]
if not matches:
    print(f'Sample {SAMPLE_LABEL} not found in database!')
else:
    sample = get_sample(matches[0]['id'])
    print(f'Sample: {sample["label"]}')
    print(f'Wafer: {sample["wafer_name"]}')
    print(f'Devices:')
    for d in sample['devices']:
        full_label = f'{sample["label"]}_D{d["device_number"]}'
        print(f'  {full_label}  (W={d["channel_w_um"]}um, L={d["channel_l_um"]}um)')

Sample: 260626_W1_P1
Wafer: 260626_W1
Devices:
  260626_W1_P1_D1  (W=10000.0um, L=20.0um)
  260626_W1_P1_D2  (W=10000.0um, L=20.0um)
  260626_W1_P1_D3  (W=10000.0um, L=20.0um)


## 2. Measurement Parameters

Set measurement conditions and HDF5 path.

In [3]:
HDF5_FILE = 'data/raw/2026-06-26_test1.hdf5'  # Relative to project root
MEASUREMENT_TYPES = ['transfer', 'output', 'stability']  # What you measured
HDF5_ABS_PATH = os.path.join(PROJECT_ROOT, HDF5_FILE)

MEASUREMENT_NOTES = ''
MEASUREMENT_PARAMS = {
    'instrument': 'Keithley 2600',
    'channels': {'GS': 'a', 'DS': 'b'},
    'V_GS_range': [-0.8, 0.4],
    'V_DS_range': [-0.8, -0.1],
}

print(f'HDF5 file: {HDF5_ABS_PATH}')
print(f'File exists: {os.path.exists(HDF5_ABS_PATH)}')

HDF5 file: /home/dibarra/Documents/egofets-project/data/raw/2026-06-26_test1.hdf5
File exists: True


In [4]:
# Register measurement session in database
for device in sample['devices']:
    dev_label = f'{sample["label"]}_D{device["device_number"]}'
    for mtype in MEASUREMENT_TYPES:
        add_measurement(
            device_id=device['id'],
            sample_id=sample['id'],
            measurement_type=mtype,
            hdf5_path=HDF5_FILE,
            notes=f'{dev_label} - {mtype}',
        )
        print(f'  Registered: {dev_label} ({mtype})')

# Mark measurement step as completed
add_step(sample['id'], 'measurement', status='completed',
         params=MEASUREMENT_PARAMS, notes=MEASUREMENT_NOTES)

print(f'\nMeasurement session recorded for {sample["label"]}.')

  Registered: 260626_W1_P1_D1 (transfer)
  Registered: 260626_W1_P1_D1 (output)
  Registered: 260626_W1_P1_D1 (stability)
  Registered: 260626_W1_P1_D2 (transfer)
  Registered: 260626_W1_P1_D2 (output)
  Registered: 260626_W1_P1_D2 (stability)
  Registered: 260626_W1_P1_D3 (transfer)
  Registered: 260626_W1_P1_D3 (output)
  Registered: 260626_W1_P1_D3 (stability)

Measurement session recorded for 260626_W1_P1.


## 3. Quick Preview of Measured Data

Verify the HDF5 file contains what you expect.

In [5]:
if os.path.exists(HDF5_ABS_PATH):
    ms = read_hdf5(HDF5_ABS_PATH)
    display(list_measurements(ms))
else:
    print(f'File not found: {HDF5_ABS_PATH}')

,name,mode,batch,device,n_points,columns
0,Batch1_Dispositivo1_Output1_00,output,Batch1,,644,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
1,Batch1_Dispositivo1_Output2_00,output,Batch1,,644,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
2,Batch1_Dispositivo1_Transfer1_00,transfer,Batch1,,162,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
3,Batch1_Dispositivo1_Transfer1_01,transfer,Batch1,,324,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
4,Batch1_Dispositivo1_Transfer2_00,transfer,Batch1,,324,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
5,Batch1_Dispositivo2_Output_00,output,Batch1,,644,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
6,Batch1_Dispositivo2_Output_01,output,Batch1,,644,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
7,Batch1_Dispositivo2_Output_02,transfer,Batch1,,324,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
8,Batch1_Dispositivo2_Output_03,output,Batch1,,644,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"
9,Batch1_Dispositivo2_Transfer1_00,transfer,Batch1,,324,"[V_DS, V_GS, I_DS, I_GS, time, curve, timestamp]"


---
### Notes
- For analyzing these measurements, see `notebooks/02_transfer_curves/` and `notebooks/03_output_curves/`
- Link to dashboard: `streamlit run dashboard/app.py`
- Store raw HDF5 files in `data/raw/`, organized by date